In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Always load from processed/ — never from raw/
file_path = os.path.join('..', 'data', 'processed', 'josaa_clean.csv')
df = pd.read_csv(file_path)

print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nExisting columns:")
for col in df.columns:
    print(f"  {col}")

Data loaded: 432,524 rows, 14 columns

Existing columns:
  institute
  program
  quota
  seat_type
  gender
  opening_rank
  closing_rank
  round
  year
  is_special_round
  institute_type
  is_pwd
  category_base
  gender_short


In [2]:
# RANK WINDOW = Closing Rank - Opening Rank
#
# Business meaning:
# A small rank window (e.g. 50) means very few students
# were admitted in that rank range — extremely competitive
# A large rank window (e.g. 5000) means seats were spread
# across a wide rank range — more accessible
#
# Example:
# IIT Delhi CSE: Opening 1, Closing 50   → window = 49  (ultra competitive)
# NIT Nagaland CE: Opening 50000, Closing 80000 → window = 30000 (very accessible)
#
# This feature will be important for ML — it captures
# demand intensity independent of absolute rank values

df['rank_window'] = df['closing_rank'] - df['opening_rank']

print("Rank Window Feature created.")
print(f"\nRank window statistics:")
print(df['rank_window'].describe().round(0))

print(f"\nSample rows:")
print(
    df[['institute', 'program', 'category_base',
        'opening_rank', 'closing_rank', 'rank_window']]
    .head(10)
    .to_string(index=False)
)

Rank Window Feature created.

Rank window statistics:
count    432524.0
mean       2868.0
std       17091.0
min           0.0
25%           0.0
50%         281.0
75%        1533.0
max      982920.0
Name: rank_window, dtype: float64

Sample rows:
                                 institute                                                                                                             program category_base  opening_rank  closing_rank  rank_window
Indian Institute of Technology Bhubaneswar                                                                 Civil Engineering (4 Years, Bachelor of Technology)          OPEN          5057          6780         1723
Indian Institute of Technology Bhubaneswar                                                                 Civil Engineering (4 Years, Bachelor of Technology)          OPEN         10078         10789          711
Indian Institute of Technology Bhubaneswar                                                                 Civil

In [3]:
# COMPETITIVENESS SCORE
#
# Problem: Raw closing ranks are not comparable across categories
# IIT CSE OPEN closes at 100, IIT CSE ST closes at 30
# Both are equally "competitive" within their category
# but raw numbers make ST look less competitive
#
# Solution: Normalize closing rank WITHIN each category and year
# Score = 1 - (rank / max_rank_in_category_year)
# Score close to 1 = very competitive (low rank)
# Score close to 0 = less competitive (high rank)
#
# This enables fair cross-category comparison for ML

# Calculate max closing rank per category per year
max_rank = (
    df.groupby(['year', 'category_base'])['closing_rank']
    .transform('max')
)

df['competitiveness_score'] = (
    1 - (df['closing_rank'] / max_rank)
).round(4)

print("Competitiveness Score feature created.")
print(f"\nScore range: {df['competitiveness_score'].min()} to {df['competitiveness_score'].max()}")
print(f"\nMean score by institute type:")
print(
    df.groupby('institute_type')['competitiveness_score']
    .mean()
    .round(3)
    .sort_values(ascending=False)
)

Competitiveness Score feature created.

Score range: 0.0 to 1.0

Mean score by institute type:
institute_type
IIT     0.987
IIIT    0.960
NIT     0.939
GFTI    0.918
Name: competitiveness_score, dtype: float64


In [4]:
# IIT GENERATION
# Old IITs (Pre-2008) vs New IITs (Post-2008)
# We established in Phase 5 that this distinction
# is analytically meaningful — the gap is widening

old_iits = [
    'Indian Institute of Technology Bombay',
    'Indian Institute of Technology Delhi',
    'Indian Institute of Technology Kanpur',
    'Indian Institute of Technology Kharagpur',
    'Indian Institute of Technology Madras',
    'Indian Institute of Technology Roorkee',
    'Indian Institute of Technology Guwahati'
]

def get_iit_generation(row):
    if row['institute_type'] != 'IIT':
        return 'Not IIT'
    elif row['institute'] in old_iits:
        return 'Old IIT'
    else:
        return 'New IIT'

df['iit_generation'] = df.apply(get_iit_generation, axis=1)

print("IIT Generation feature created.")
print(df['iit_generation'].value_counts())

IIT Generation feature created.
iit_generation
Not IIT    313513
New IIT     61612
Old IIT     57399
Name: count, dtype: int64


In [5]:
# NIT REGION
# We discovered in Phase 5 that NIT location strongly predicts competitiveness
# We group NITs into 4 geographic regions
# This captures the location premium effect as a feature

nit_region_map = {
    # South India
    'National Institute of Technology Karnataka, Surathkal'      : 'South',
    'National Institute of Technology, Tiruchirappalli'          : 'South',
    'National Institute of Technology Calicut'                   : 'South',
    'National Institute of Technology, Andhra Pradesh'           : 'South',
    'National Institute of Technology Puducherry'                : 'South',
    'National Institute of Technology, Warangal'                 : 'South',
    'National Institute of Technology Goa'                       : 'South',

    # North India
    'Malaviya National Institute of Technology Jaipur'           : 'North',
    'National Institute of Technology Delhi'                     : 'North',
    'Maulana Azad National Institute of Technology Bhopal'       : 'North',
    'Motilal Nehru National Institute of Technology Allahabad'   : 'North',
    'National Institute of Technology Kurukshetra'               : 'North',
    'National Institute of Technology, Kurukshetra'              : 'North',
    'Dr. B R Ambedkar National Institute of Technology, Jalandhar': 'North',
    'National Institute of Technology Hamirpur'                  : 'North',
    'National Institute of Technology Srinagar'                  : 'North',
    'National Institute of Technology, Srinagar'                 : 'North',
    'National Institute of Technology, Uttarakhand'              : 'North',
    'National Institute of Technology Patna'                     : 'North',
    'National Institute of Technology, Jamshedpur'               : 'North',

    # West and Central India
    'Visvesvaraya National Institute of Technology, Nagpur'      : 'West',
    'Sardar Vallabhbhai National Institute of Technology, Surat' : 'West',
    'National Institute of Technology Raipur'                    : 'West',
    'National Institute of Technology, Raipur'                   : 'West',

    # East and Northeast India
    'National Institute of Technology Rourkela'                  : 'East',
    'National Institute of Technology, Rourkela'                 : 'East',
    'National Institute of Technology Durgapur'                  : 'East',
    'National Institute of Technology Silchar'                   : 'East',
    'National Institute of Technology, Silchar'                  : 'East',
    'National Institute of Technology Agartala'                  : 'Northeast',
    'National Institute of Technology, Manipur'                  : 'Northeast',
    'National Institute of Technology Meghalaya'                 : 'Northeast',
    'National Institute of Technology, Mizoram'                  : 'Northeast',
    'National Institute of Technology Nagaland'                  : 'Northeast',
    'National Institute of Technology Sikkim'                    : 'Northeast',
    'National Institute of Technology Arunachal Pradesh'         : 'Northeast',
}

def get_nit_region(row):
    if row['institute_type'] != 'NIT':
        return 'Not NIT'
    return nit_region_map.get(row['institute'], 'Unknown')

df['nit_region'] = df.apply(get_nit_region, axis=1)

print("NIT Region feature created.")
print(df['nit_region'].value_counts())

# Check for any unknowns
unknown_nits = df[
    (df['institute_type'] == 'NIT') &
    (df['nit_region'] == 'Unknown')
]['institute'].unique()
print(f"\nUnknown NITs: {len(unknown_nits)}")
for n in unknown_nits:
    print(f"  {n}")

NIT Region feature created.
nit_region
Not NIT      198649
North         90562
South         57077
East          32066
West          29188
Northeast     24982
Name: count, dtype: int64

Unknown NITs: 0


In [16]:
# PROGRAM CATEGORY
# Groups 326 individual programs into broad families
# This reduces cardinality for ML encoding
# and enables branch-family level analysis
#
# Families: CS_AI, Electrical, Mechanical, Civil,
#           Chemical, Science, Interdisciplinary, Other

import re

def get_program_category(program):
    p = program.upper()

    if any(k in p for k in [
        'COMPUTER SCIENCE', 'INFORMATION TECHNOLOGY',
        'ARTIFICIAL INTELLIGENCE', 'DATA SCIENCE',
        'MACHINE LEARNING', 'COMPUTING', 'SOFTWARE',
        'COMPUTER ENGINEERING'
    ]):
        return 'CS'

    elif any(k in p for k in [
        'ELECTRICAL', 'ELECTRONICS', 'COMMUNICATION',
        'INSTRUMENTATION', 'VLSI', 'SIGNAL'
    ]):
        return 'Electrical_Electronics'

    elif any(k in p for k in [
        'MECHANICAL', 'PRODUCTION', 'INDUSTRIAL',
        'MANUFACTURING', 'MECHATRONICS', 'AEROSPACE',
        'AUTOMOTIVE', 'ROBOTICS', 'DESIGN'
    ]):
        return 'Mechanical'

    elif any(k in p for k in [
        'CIVIL', 'STRUCTURAL', 'CONSTRUCTION',
        'ENVIRONMENTAL', 'TRANSPORTATION', 'GEOTECHNICAL'
    ]):
        return 'Civil'

    elif any(k in p for k in [
        'CHEMICAL', 'BIOCHEMICAL', 'PETROCHEMICAL',
        'POLYMER', 'PROCESS', 'FOOD'
    ]):
        return 'Chemical'

    elif any(k in p for k in [
        'METALLURG', 'MATERIALS', 'MINING',
        'CERAMIC', 'MINERAL', 'FOUNDRY'
    ]):
        return 'Materials_Mining'

    elif any(k in p for k in [
        'PHYSICS', 'CHEMISTRY', 'MATHEMATICS',
        'BIOLOGY', 'SCIENCE', 'LIFE SCIENCE',
        'ENGINEERING SCIENCE', 'ENERGY'
    ]):
        return 'Science_Interdisciplinary'

    elif any(k in p for k in [
        'ARCHITECTURE', 'PLANNING'
    ]):
        return 'Architecture_Planning'

    else:
        return 'Other'

df['program_category'] = df['program'].apply(get_program_category)

print("Program Category feature created.")
print(df['program_category'].value_counts())

print(f"\nSample CS programs:")
print(
    df[df['program_category'] == 'CS']['program']
    .unique()[:8]
)

Program Category feature created.
program_category
Electrical_Electronics       103943
CS                           100617
Mechanical                    64246
Civil                         42700
Chemical                      32812
Materials_Mining              30309
Science_Interdisciplinary     24740
Other                         19847
Architecture_Planning         13310
Name: count, dtype: int64

Sample CS programs:
<StringArray>
[                                         'Computer Science and Engineering (4 Years, Bachelor of Technology)',
                                      'Mathematics and Scientific Computing (4 Years, Bachelor of Technology)',
                              'Artificial Intelligence and Machine Learning (4 Years, Bachelor of Technology)',
                                                    'Information Technology (4 Years, Bachelor of Technology)',
                                           'Mathematics and Computing (5 Years, Integrated Master of Science)',
    

In [9]:
# YoY CLOSING RANK CHANGE
# For each institute-program-category-gender combination
# we calculate how the closing rank changed from previous year
# Negative = became more competitive
# Positive = became less competitive
#
# This is a critical ML feature — it captures momentum
# A program consistently getting harder is a strong prediction signal

# Sort first to ensure correct year ordering
df = df.sort_values(['institute', 'program', 'category_base',
                     'gender_short', 'quota', 'round', 'year'])

# Calculate YoY change within each group
df['yoy_closing_rank_change'] = (
    df.groupby([
        'institute', 'program',
        'category_base', 'gender_short', 'quota'
    ])['closing_rank']
    .diff()   # diff() subtracts previous row value from current
)

# Count non-null values to verify
non_null = df['yoy_closing_rank_change'].notna().sum()
print(f"YoY change feature created.")
print(f"Non-null values: {non_null:,}")
print(f"Null values (first year of each group): {df['yoy_closing_rank_change'].isna().sum():,}")
print(f"\nYoY change statistics:")
print(df['yoy_closing_rank_change'].describe().round(0))

YoY change feature created.
Non-null values: 417,059
Null values (first year of each group): 15,465

YoY change statistics:
count    417059.0
mean         93.0
std       20888.0
min     -782139.0
25%        -769.0
50%          48.0
75%        1107.0
max      779502.0
Name: yoy_closing_rank_change, dtype: float64


In [10]:
# IS NEW AGE PROGRAM
# Binary flag for programs in high-demand emerging fields
# AI, ML, Data Science, Robotics, Cybersecurity etc.
# These programs showed strongest competitiveness gains in Phase 5
# This flag enables quick filtering in dashboard and ML

new_age_keywords = [
    'Artificial Intelligence',
    'Machine Learning',
    'Data Science',
    'Robotics',
    'Cybersecurity',
    'Internet of Things',
    'Blockchain',
    'Cloud Computing',
    'Bioinformatics',
    'Computational',
]

df['is_new_age_program'] = df['program'].apply(
    lambda x: any(kw.lower() in x.lower() for kw in new_age_keywords)
)

print("New Age Program flag created.")
print(df['is_new_age_program'].value_counts())
print(f"\nNew age programs found:")
print(
    df[df['is_new_age_program'] == True]['program']
    .unique()
)

New Age Program flag created.
is_new_age_program
False    416590
True      15934
Name: count, dtype: int64

New age programs found:
<StringArray>
[                                                                                'Artificial Intelligence and Machine Learning (4 Years, Bachelor of Technology)',
                                                                                  'Quantitative Economics & Data Science (5 Years, Integrated Master of Science)',
                                                                    'Computer Science Engineering (Artificial Intelligence) (4 Years, B. Tech / B. Tech (Hons.))',
                                                                               'Computer Science Engineering (Data Science) (4 Years, B. Tech / B. Tech (Hons.))',
                                                                                                 'Data Science and Engineering (4 Years, Bachelor of Technology)',
                                       

In [18]:
# IS TOP TIER COLLEGE
# Binary flag for colleges that consistently appeared
# in top competitive rankings from Phase 5
# Useful for quick filtering and as ML feature

top_tier_colleges = [
    # Top 7 Old IITs
    'Indian Institute of Technology Bombay',
    'Indian Institute of Technology Delhi',
    'Indian Institute of Technology Kanpur',
    'Indian Institute of Technology Kharagpur',
    'Indian Institute of Technology Madras',
    'Indian Institute of Technology Roorkee',
    'Indian Institute of Technology Guwahati',

    # Top performing newer IITs
    'Indian Institute of Technology Hyderabad',
    'Indian Institute of Technology Indore',
    'Indian Institute of Technology Gandhinagar',

    # Top NITs
    'National Institute of Technology Karnataka, Surathkal',
    'National Institute of Technology, Tiruchirappalli',
    'Malaviya National Institute of Technology Jaipur',
    'National Institute of Technology Delhi',
    'Maulana Azad National Institute of Technology Bhopal',

    # Top IIITs — fastest rising in Phase 5
    'Indian Institute of Information Technology, Allahabad',
    'Atal Bihari Vajpayee Indian Institute of Information Technology & Management Gwalior',
    'Indian Institute of Information Technology Lucknow',
]

df['is_top_tier_college'] = df['institute'].isin(top_tier_colleges)

print("Top Tier College flag created.")
print(df['is_top_tier_college'].value_counts())
print(f"\nTop tier college rows: {df['is_top_tier_college'].sum():,}")

Top Tier College flag created.
is_top_tier_college
False    314625
True     117899
Name: count, dtype: int64

Top tier college rows: 117,899


In [12]:
# SEAT ACCESSIBILITY LABEL
# Human readable label based on closing rank
# Useful for dashboard filters and storytelling
#
# Thresholds based on our Phase 5 analysis:
# Very High Competition : closing rank < 5,000
# High Competition      : 5,000 to 15,000
# Moderate Competition  : 15,000 to 40,000
# Low Competition       : > 40,000

def get_accessibility_label(rank):
    if rank < 5000:
        return 'Very High Competition'
    elif rank < 15000:
        return 'High Competition'
    elif rank < 40000:
        return 'Moderate Competition'
    else:
        return 'Low Competition'

df['accessibility_label'] = df['closing_rank'].apply(get_accessibility_label)

print("Accessibility Label feature created.")
print(df['accessibility_label'].value_counts())

# Verify distribution makes sense per institute type
print("\nAccessibility label by institute type:")
print(
    df.groupby(['institute_type', 'accessibility_label'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

Accessibility Label feature created.
accessibility_label
Very High Competition    226362
High Competition         116490
Moderate Competition      59216
Low Competition           30456
Name: count, dtype: int64

Accessibility label by institute type:
accessibility_label  High Competition  Low Competition  Moderate Competition  Very High Competition
institute_type                                                                                     
GFTI                            16514             7817                  9368                  12374
IIIT                            11691              829                  5284                  15761
IIT                             21293                0                  4144                  93574
NIT                             66992            21810                 40420                 104653


In [19]:
# Print complete feature list
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"\nTotal columns : {len(df.columns)}")
print(f"Total rows    : {len(df):,}")

print(f"\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

# Check for any unexpected nulls in new features
new_features = [
    'rank_window', 'competitiveness_score', 'iit_generation',
    'nit_region', 'program_category', 'yoy_closing_rank_change',
    'is_new_age_program', 'is_top_tier_college', 'accessibility_label'
]

print(f"\nNull check on new features:")
for feat in new_features:
    nulls = df[feat].isna().sum()
    print(f"  {feat:35} : {nulls:,} nulls")

# Save enriched dataset
output_path = os.path.join('..', 'data', 'processed', 'josaa_featured.csv')
df.to_csv(output_path, index=False)
print(f"\nFeatured dataset saved: {output_path}")
print(f"Shape: {df.shape}")

FEATURE ENGINEERING COMPLETE

Total columns : 23
Total rows    : 432,524

All columns:
   1. institute
   2. program
   3. quota
   4. seat_type
   5. gender
   6. opening_rank
   7. closing_rank
   8. round
   9. year
  10. is_special_round
  11. institute_type
  12. is_pwd
  13. category_base
  14. gender_short
  15. rank_window
  16. competitiveness_score
  17. iit_generation
  18. nit_region
  19. program_category
  20. yoy_closing_rank_change
  21. is_new_age_program
  22. is_top_tier_college
  23. accessibility_label

Null check on new features:
  rank_window                         : 0 nulls
  competitiveness_score               : 0 nulls
  iit_generation                      : 0 nulls
  nit_region                          : 0 nulls
  program_category                    : 0 nulls
  yoy_closing_rank_change             : 15,465 nulls
  is_new_age_program                  : 0 nulls
  is_top_tier_college                 : 0 nulls
  accessibility_label                 : 0 nulls

Feat

In [23]:
# FACT TABLE: josaa_facts
# One row per cutoff record
# Contains all numerical measures and foreign keys

fact_table = df[[
    'institute',
    'program',
    'quota',
    'seat_type',
    'category_base',
    'gender_short',
    'round',
    'year',
    'opening_rank',
    'closing_rank',
    'rank_window',
    'competitiveness_score',
    'yoy_closing_rank_change',
    'is_special_round',
    'is_pwd',
    'is_new_age_program',
    'is_top_tier_college',
    'accessibility_label'
]].copy()

output = os.path.join('..', 'data', 'processed', 'powerbi_facts.csv')
fact_table.to_csv(output, index=False)
print(f"Fact table saved: {output}")
print(f"Shape: {fact_table.shape}")

Fact table saved: ..\data\processed\powerbi_facts.csv
Shape: (432524, 18)


In [25]:
# DIMENSION TABLE: dim_institute
# One row per institute
# Contains all institute attributes

dim_institute = df[[
    'institute',
    'institute_type',
    'iit_generation',
    'nit_region',
    'is_top_tier_college'
]].drop_duplicates(subset=['institute']).copy()

# Add competitiveness rank per institute
inst_rank = (
    df.groupby('institute')['closing_rank']
    .median()
    .reset_index()
    .rename(columns={'closing_rank': 'median_closing_rank'})
    .sort_values('median_closing_rank')
)
inst_rank['overall_rank'] = range(1, len(inst_rank) + 1)

dim_institute = dim_institute.merge(inst_rank, on='institute', how='left')

output = os.path.join('..', 'data', 'processed', 'powerbi_dim_institute.csv')
dim_institute.to_csv(output, index=False)
print(f"Institute dimension saved: {output}")
print(f"Shape: {dim_institute.shape}")
print(f"\nSample:")
print(dim_institute.head(5).to_string(index=False))

Institute dimension saved: ..\data\processed\powerbi_dim_institute.csv
Shape: (136, 7)

Sample:
                                                                           institute institute_type iit_generation nit_region  is_top_tier_college  median_closing_rank  overall_rank
                                                           Assam University, Silchar           GFTI        Not IIT    Not NIT                False              19277.0           131
Atal Bihari Vajpayee Indian Institute of Information Technology & Management Gwalior           IIIT        Not IIT    Not NIT                 True               3366.0            33
                                   Birla Institute of Technology, Deoghar Off-Campus           GFTI        Not IIT    Not NIT                False              21226.0           134
                                        Birla Institute of Technology, Mesra, Ranchi           GFTI        Not IIT    Not NIT                False               9585.0         

In [26]:
# DIMENSION TABLE: dim_program
# One row per program
# Contains program attributes and competitiveness metrics

dim_program = df[[
    'program',
    'program_category',
    'is_new_age_program'
]].drop_duplicates(subset=['program']).copy()

prog_rank = (
    df.groupby('program')['closing_rank']
    .median()
    .reset_index()
    .rename(columns={'closing_rank': 'median_closing_rank'})
    .sort_values('median_closing_rank')
)
prog_rank['program_rank'] = range(1, len(prog_rank) + 1)

dim_program = dim_program.merge(prog_rank, on='program', how='left')

output = os.path.join('..', 'data', 'processed', 'powerbi_dim_program.csv')
dim_program.to_csv(output, index=False)
print(f"Program dimension saved: {output}")
print(f"Shape: {dim_program.shape}")

Program dimension saved: ..\data\processed\powerbi_dim_program.csv
Shape: (331, 5)


In [27]:
# DIMENSION TABLE: dim_year
# One row per year
# Contains year-level context flags

dim_year = pd.DataFrame({
    'year'             : [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'is_covid_year'    : [False, False, True, True, False, False, False, False],
    'total_rounds'     : [7, 7, 6, 6, 6, 6, 5, 6],
    'ews_available'    : [False, True, True, True, True, True, True, True],
    'year_label'       : [
        '2018', '2019 (Peak)', '2020 (COVID)',
        '2021 (COVID)', '2022', '2023',
        '2024 (5 Rounds)', '2025'
    ]
})

output = os.path.join('..', 'data', 'processed', 'powerbi_dim_year.csv')
dim_year.to_csv(output, index=False)
print(f"Year dimension saved: {output}")
print(dim_year.to_string(index=False))

Year dimension saved: ..\data\processed\powerbi_dim_year.csv
 year  is_covid_year  total_rounds  ews_available      year_label
 2018          False             7          False            2018
 2019          False             7           True     2019 (Peak)
 2020           True             6           True    2020 (COVID)
 2021           True             6           True    2021 (COVID)
 2022          False             6           True            2022
 2023          False             6           True            2023
 2024          False             5           True 2024 (5 Rounds)
 2025          False             6           True            2025


In [28]:
# Pre-aggregated tables for Power BI KPI cards
# Power BI renders faster with pre-aggregated data

# Institute type summary
inst_type_summary = (
    df.groupby('institute_type').agg(
        total_institutes=('institute', 'nunique'),
        total_programs=('program', 'nunique'),
        median_closing_rank=('closing_rank', 'median'),
        mean_closing_rank=('closing_rank', 'mean'),
        total_records=('closing_rank', 'count')
    ).round(0).reset_index()
)

output = os.path.join('..', 'data', 'processed', 'powerbi_inst_type_summary.csv')
inst_type_summary.to_csv(output, index=False)
print("Institute type summary saved.")
print(inst_type_summary.to_string(index=False))

# Category summary
cat_summary = (
    df.groupby('category_base').agg(
        median_closing_rank=('closing_rank', 'median'),
        mean_closing_rank=('closing_rank', 'mean'),
        total_records=('closing_rank', 'count')
    ).round(0).reset_index()
)

output = os.path.join('..', 'data', 'processed', 'powerbi_cat_summary.csv')
cat_summary.to_csv(output, index=False)
print("\nCategory summary saved.")

# Yearly trend summary
yearly_summary = (
    df.groupby(['year', 'institute_type']).agg(
        median_closing_rank=('closing_rank', 'median'),
        total_records=('closing_rank', 'count')
    ).round(0).reset_index()
)

output = os.path.join('..', 'data', 'processed', 'powerbi_yearly_summary.csv')
yearly_summary.to_csv(output, index=False)
print("Yearly summary saved.")

print("\nAll Power BI files exported successfully.")
print(f"\nFiles in processed folder:")
for f in os.listdir(os.path.join('..', 'data', 'processed')):
    size = os.path.getsize(
        os.path.join('..', 'data', 'processed', f)
    ) / 1024
    print(f"  {f:45} {size:8.1f} KB")

Institute type summary saved.
institute_type  total_institutes  total_programs  median_closing_rank  mean_closing_rank  total_records
          GFTI                52              96              10762.0            21821.0          46073
          IIIT                30              64               5428.0             9318.0          33565
           IIT                23             190               1958.0             3501.0         119011
           NIT                31              91               6079.0            18069.0         233875

Category summary saved.
Yearly summary saved.

All Power BI files exported successfully.

Files in processed folder:
  architecture_colleges.csv                         35.6 KB
  josaa_clean.csv                                80028.2 KB
  josaa_featured.csv                            112454.2 KB
  powerbi_cat_summary.csv                            0.2 KB
  powerbi_dim_institute.csv                         12.4 KB
  powerbi_dim_program.csv       